# 05 — The recovery grid

Forty full-posterior fits over a factorial design: two panel lengths, two
collinearity levels, two specifications, five seeds. This notebook fits nothing;
it reads what `scripts/run_recovery.py` produced.

Both length arms use the *simulated* extended baseline, including the 85-week arm,
so that length is the only quantity changing along the length axis. Only the
headline fit in notebook 03 uses the real Olist baseline.

In [ ]:
import json
import warnings

import pandas as pd

from athar import paths
from athar.provenance import read_metric

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

METRICS = paths.metrics_dir()
PROCESSED = paths.processed_dir()


def show(frame, caption=""):
    if caption:
        print(caption)
    print(frame.to_string(index=False))
    print()

In [ ]:
recovery = read_metric("recovery", METRICS)
print(json.dumps(recovery["design"], indent=2))
print()
print(json.dumps(recovery["convergence"], indent=2))

## Coverage, and why coverage alone is not enough

Coverage is the property a Bayesian model claims: the truth should fall inside the
89% interval about 89% of the time. But an interval can achieve perfect coverage by
being uselessly wide, so interval width is reported beside it. A cell with coverage
1.00 and a median error of twenty is not a success.

In [ ]:
rows = []
for name, slice_ in recovery["slices"].items():
    for quantity in ("average_roi", "marginal_roi"):
        block = slice_[quantity]
        if block.get("converged", 0) == 0:
            rows.append({"cell": name, "quantity": quantity, "converged": 0})
            continue
        rows.append(
            {
                "cell": name,
                "quantity": quantity,
                "converged": block["converged"],
                "coverage": block["coverage_rate"],
                "median |rel err|": round(block["median_absolute_relative_error"], 3),
                "mean interval width": round(block["mean_interval_width"], 3),
            }
        )
show(pd.DataFrame(rows).sort_values(["quantity", "cell"]), "Every cell of the grid")

## Does the choice of convergence rule change the conclusion

The primary rule uses a divergence rate; the strict rule demands zero divergences
and rejected almost every fit when it was first applied. Both are computed for
every cell so the effect of the choice is visible rather than asserted.

In [ ]:
rows = []
for name, slice_ in recovery["slices"].items():
    primary = slice_["average_roi"]
    strict = slice_.get("average_roi_strict_convergence", {})
    rows.append(
        {
            "cell": name,
            "converged (rate rule)": primary.get("converged", 0),
            "coverage (rate rule)": primary.get("coverage_rate"),
            "converged (strict)": strict.get("converged", 0),
            "coverage (strict)": strict.get("coverage_rate"),
        }
    )
show(pd.DataFrame(rows))
print(recovery["convergence"]["rule_note"])

## Error against the identification diagnostics

If recovery error tracks the condition number and the media signal share, then the
failures are a property of the *design* rather than of the method — which is the
more useful thing for a practitioner to know, because a design is something they
control.

In [ ]:
fits = pd.DataFrame(
    [
        {
            "weeks": f["weeks"],
            "collinearity": f["collinearity"],
            "specification": f["specification"],
            "converged": f["diagnostics"]["passed"],
            "condition_number": f["identification"]["condition_number"],
            "media_signal": f["identification"]["media_share_of_detrended_variance"],
            "coverage": f["average_roi"]["summary"]["coverage_rate"],
            "median_abs_rel_error": f["average_roi"]["summary"]["median_absolute_relative_error"],
            "mean_interval_width": f["average_roi"]["summary"]["mean_interval_width"],
        }
        for f in recovery["fits"]
    ]
)
usable = fits[fits["converged"]]
show(
    usable.groupby(["specification", "collinearity", "weeks"])
    .agg(
        fits=("coverage", "size"),
        coverage=("coverage", "mean"),
        median_error=("median_abs_rel_error", "median"),
        interval_width=("mean_interval_width", "mean"),
        condition=("condition_number", "mean"),
        signal=("media_signal", "mean"),
    )
    .round(3)
    .reset_index()
)

## Per channel

Which channels are recoverable is not uniform, and the pattern is worth more than
the average: a channel that is never recovered is a channel a media-mix model
should not be used to budget.

In [ ]:
rows = []
for name, slice_ in recovery["slices"].items():
    block = slice_["average_roi"]
    for channel, entry in block.get("per_channel", {}).items():
        rows.append({"cell": name, "channel": channel, **entry})
if rows:
    per_channel = pd.DataFrame(rows)
    show(
        per_channel.groupby("channel")
        .agg(
            true=("true", "first"),
            median_estimate=("median_estimate", "median"),
            coverage=("coverage_rate", "mean"),
            median_rel_error=("median_relative_error", "median"),
        )
        .round(3)
        .reset_index(),
        "Across every converged cell",
    )